# Realism validation -- rewritten to match `01_dataset_generation_v18.ipynb`

The original validation notebook was built against a **different dataset format** than what
the v18 generator actually produces, so almost every cell was silently checking the wrong
thing. This rewrite fixes that. What changed, and why:

| # | Old validator assumed | v18 generator actually produces |
|---|---|---|
| 1 | A single `.npz` file with keys `eeg`/`ieeg`/`fs`/`onsets`/`ied_onset_samples` | **Five CSV files** (`continuous_dataset_15.csv` etc., CELLS 21-22) with columns `subject_id, time_sec, ied_flag, <20 scalp channels>, <12 FO channels>` |
| 2 | `fs = 256 Hz` | **`fs = 200 Hz`** (CELL 1) |
| 3 | `20 minutes` / `1200 s` per subject | **`900 s` (15 min)** per subject (CELL 1, `duration_sec`) |
| 4 | `500` IEDs/subject expected | **`100`** IEDs/subject, exactly (CELL 1, `n_ied`) -> ~6.7/min over 900s |
| 5 | Signals stored in **volts** (`EEG_UV = EEG * 1e6`) | Signals are calibrated directly to **microvolts** already (`EEG_BG_TARGET_RANGE=(12,20)`, `IEEG_BG_TARGET_RANGE=(70,130)` in CELL 1/20) -- multiplying by 1e6 would have inflated everything 6 orders of magnitude |
| 6 | Generic/undefined channel names | Exact channel lists from CELL 1 (`scalp_channels`) and CELL 15 (`ieeg_channels = LFO1..LFO6, RFO1..RFO6`) |
| 7 | IED amplitude target `500-4000 µV`, attenuation `15-30x` | Generator's own stated targets (from its changelog cells): **FO peak 200-1000 µV**, **FO/EEG peak ratio 10-40**, **EEG FWHM 15-40 ms**, **FO FWHM 10-30 ms**, **FO L/R lateralisation >= 5**, **propagation lag 5-30 ms**, **IED rate 2-10/min**, **amplitude CV 0.25-0.8**, **relative alpha 0.15-0.30**, **scalp AUC 0.50-0.75** |
| 8 | A handful of duplicate/dead-end debug cells (checking `data["onsets"]`, `data["ied_onset_samples"]`, `EXPECTED_IEDS_PER_SUBJECT=500` twice) | Consolidated into one clean loading step |

Everything below re-derives its reference targets from the generator's own CELL 1 constants
and changelog commentary, not from a mismatched external assumption. Edit **CELL 1** below to
point `CSV_PATH` at your actual output file, then run top to bottom.


In [1]:
# ============================================================
# CELL 1 -- Configuration (must match 01_dataset_generation_v18.ipynb CELL 1)
# ============================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import signal, stats
from scipy.signal import correlate

# ---- point this at your generated continuous dataset --------------------------------------
# CELL 21 of the generator writes 'data/continuous_dataset_15.csv' by default.
CSV_PATH = "data/continuous_dataset_15.csv"

# ---- generator constants (CELL 1 of 01_dataset_generation_v18.ipynb) ----------------------
FS = 200.0
DURATION_SEC_EXPECTED = 900.0            # 15 min/subject
N_SUBJECTS_EXPECTED = 18
N_EEG = 20
N_FO = 12
N_IED_EXPECTED_PER_SUBJECT = 100         # exact, per CELL 1 comment ("v13 removes the jitter")
EEG_BG_TARGET_RANGE = (12.0, 20.0)       # uV, CELL 1 EEG_BG_TARGET_RANGE
IEEG_BG_TARGET_RANGE = (70.0, 130.0)     # uV, CELL 1 IEEG_BG_TARGET_RANGE

SCALP_CHANNELS = ['Fp1', 'Fp2', 'F7', 'F3', 'Fz', 'F4', 'F8', 'T7', 'C3', 'Cz',
                   'C4', 'T8', 'P7', 'P3', 'Pz', 'P4', 'P8', 'O1', 'Oz', 'O2']
IEEG_CHANNELS = [f'{h}FO{i}' for h in ('L', 'R') for i in range(1, 7)]  # LFO1..LFO6, RFO1..RFO6

SCALP_HEMI = {'Fp1': 'L', 'F7': 'L', 'F3': 'L', 'T7': 'L', 'C3': 'L', 'P7': 'L', 'P3': 'L', 'O1': 'L',
              'Fp2': 'R', 'F8': 'R', 'F4': 'R', 'T8': 'R', 'C4': 'R', 'P8': 'R', 'P4': 'R', 'O2': 'R',
              'Fz': 'mid', 'Cz': 'mid', 'Pz': 'mid', 'Oz': 'mid'}
IEEG_HEMI = {ch: ch[0] for ch in IEEG_CHANNELS}   # 'L'/'R' from the channel name itself

BANDS = {"delta": (1, 4), "theta": (4, 8), "alpha": (8, 13), "beta": (13, 30), "gamma": (30, 80)}

# ---- generator-declared validation targets (sourced from the v11-v18 changelog cells) -----
TARGETS = dict(
    ied_rate_per_min=(2.0, 10.0),                       # v13 changelog
    eeg_fwhm_ms=(15.0, 40.0),                            # v11 changelog
    fo_fwhm_ms=(10.0, 30.0),                             # v11 changelog
    propagation_lag_ms=(5.0, 30.0),                      # v11 changelog
    fo_lateralisation_ratio=(5.0, np.inf),               # v17/v18 changelog (>=5)
    fo_eeg_peak_ratio=(10.0, 40.0),                      # v16 changelog
    fo_peak_amp_uv=(200.0, 1000.0),                      # v17 changelog
    ied_amp_cv=(0.25, 0.80),                             # v13/v14 changelog
    eeg_relative_alpha=(0.15, 0.30),                     # v14 changelog
    scalp_auc=(0.50, 0.75),                              # v14/v15 changelog
    fo_line_length_ratio=(1.8, 8.0),                     # v18 changelog
    fo_ersp_gain=(3.0, np.inf),                          # v18 changelog (>=3)
)

# NumPy 2.0 removed np.trapz in favour of np.trapezoid -- support both.
_trapz = getattr(np, "trapezoid", None) or np.trapz

print("Config loaded. Expecting:")
print(f"  fs={FS} Hz, duration={DURATION_SEC_EXPECTED}s ({DURATION_SEC_EXPECTED/60:.0f} min), "
      f"{N_SUBJECTS_EXPECTED} subjects, {N_EEG} EEG ch, {N_FO} FO ch, "
      f"{N_IED_EXPECTED_PER_SUBJECT} IEDs/subject")


Config loaded. Expecting:
  fs=200.0 Hz, duration=900.0s (15 min), 18 subjects, 20 EEG ch, 12 FO ch, 100 IEDs/subject


In [2]:
# ============================================================
# CELL 2 -- Load the continuous CSV and reshape to (subject, channel, sample) arrays
# ============================================================
# Values in the CSV are already in microvolts (the generator calibrates directly to
# EEG_BG_TARGET_RANGE / IEEG_BG_TARGET_RANGE, both already expressed in uV -- see CELL 20
# of the generator). No V->uV conversion is applied here.
#
# IED onsets are recovered from the 'ied_flag' column (set to 1 at the exact onset sample
# for every event in d['ied_times'] -- see CELL 21 of the generator), not guessed from a
# separate onsets array that this CSV format does not contain.

raw = pd.read_csv(CSV_PATH)
required_cols = {"subject_id", "time_sec", "ied_flag"}
missing = required_cols - set(raw.columns)
if missing:
    raise KeyError(f"Expected columns missing from {CSV_PATH}: {missing}")

missing_eeg = [c for c in SCALP_CHANNELS if c not in raw.columns]
missing_fo = [c for c in IEEG_CHANNELS if c not in raw.columns]
if missing_eeg or missing_fo:
    raise KeyError(f"Channel columns missing. EEG missing={missing_eeg} FO missing={missing_fo}")

subject_ids = sorted(raw["subject_id"].unique())
n_subjects_found = len(subject_ids)
groups = dict(tuple(raw.groupby("subject_id")))

n_samples_expected = int(round(DURATION_SEC_EXPECTED * FS))
n_samples_found = groups[subject_ids[0]].shape[0]

EEG = np.zeros((n_subjects_found, N_EEG, n_samples_found), dtype=np.float64)
FO = np.zeros((n_subjects_found, N_FO, n_samples_found), dtype=np.float64)
IED_ONSETS = []   # list of 1D int arrays, one per subject, in the same order as subject_ids

for i, sid in enumerate(subject_ids):
    g = groups[sid].sort_values("time_sec")
    if g.shape[0] != n_samples_found:
        print(f"WARNING: subject {sid} has {g.shape[0]} rows, expected {n_samples_found}")
    EEG[i] = g[SCALP_CHANNELS].to_numpy().T
    FO[i] = g[IEEG_CHANNELS].to_numpy().T
    onsets_i = np.flatnonzero(g["ied_flag"].to_numpy() == 1)
    IED_ONSETS.append(onsets_i.astype(np.int64))

EEG_UV = EEG          # already microvolts -- kept as a separate name for parity with the
FO_UV = FO            # rest of this notebook's variable-naming convention

print(f"Loaded {CSV_PATH}")
print(f"Subjects found       : {n_subjects_found} (expected {N_SUBJECTS_EXPECTED})")
print(f"Samples/subject      : {n_samples_found} (expected {n_samples_expected})")
print(f"Duration/subject     : {n_samples_found/FS/60:.2f} min (expected {DURATION_SEC_EXPECTED/60:.0f} min)")
print(f"EEG array shape      : {EEG.shape}")
print(f"FO array shape       : {FO.shape}")
print(f"IED count per subject: {[len(x) for x in IED_ONSETS]}")


Loaded data/continuous_dataset_15.csv
Subjects found       : 18 (expected 18)
Samples/subject      : 180000 (expected 180000)
Duration/subject     : 15.00 min (expected 15 min)
EEG array shape      : (18, 20, 180000)
FO array shape       : (18, 12, 180000)
IED count per subject: [100, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100]


In [3]:
# ============================================================
# CELL 3 -- Signal integrity + structural checks
# ============================================================
print("\n" + "=" * 100)
print("CELL 3 -- SIGNAL INTEGRITY & STRUCTURE")
print("=" * 100)

def integrity_check(X, name):
    nan_count = int(np.isnan(X).sum())
    inf_count = int(np.isinf(X).sum())
    channel_sd = np.std(X, axis=2)
    dead_channels = int(np.sum(channel_sd < 1e-9))
    print(f"\n{name}")
    print(f"{'Parameter':30s}{'Your value':20s}{'Reference':25s}Status")
    print("-" * 90)
    for label, val, ref in [("NaN count", nan_count, "0"), ("Inf count", inf_count, "0"),
                             ("Dead channels", dead_channels, "0")]:
        status = "PASS" if val == 0 else "FAIL"
        print(f"{label:30s}{val:<20d}{ref:25s}{status}")
    return {"nan": nan_count, "inf": inf_count, "dead": dead_channels}

EEG_integrity = integrity_check(EEG, "SCALP EEG")
FO_integrity = integrity_check(FO, "FO-iEEG")

print(f"\n{'Structural check':30s}{'Your value':20s}{'Reference':25s}Status")
print("-" * 90)
checks = [
    ("Subjects", n_subjects_found, N_SUBJECTS_EXPECTED),
    ("EEG channels", EEG.shape[1], N_EEG),
    ("FO channels", FO.shape[1], N_FO),
]
for label, val, ref in checks:
    status = "PASS" if val == ref else "FAIL"
    print(f"{label:30s}{val:<20}{str(ref):25s}{status}")
dur_min = n_samples_found / FS / 60
status = "PASS" if abs(dur_min - DURATION_SEC_EXPECTED / 60) < 0.05 else "FAIL"
print(f"{'Duration (min)':30s}{dur_min:<20.2f}{DURATION_SEC_EXPECTED/60:<25.1f}{status}")



CELL 3 -- SIGNAL INTEGRITY & STRUCTURE

SCALP EEG
Parameter                     Your value          Reference                Status
------------------------------------------------------------------------------------------
NaN count                     0                   0                        PASS
Inf count                     0                   0                        PASS
Dead channels                 0                   0                        PASS

FO-iEEG
Parameter                     Your value          Reference                Status
------------------------------------------------------------------------------------------
NaN count                     0                   0                        PASS
Inf count                     0                   0                        PASS
Dead channels                 0                   0                        PASS

Structural check              Your value          Reference                Status
-------------------------------

In [4]:
# ============================================================
# CELL 4 -- Background RMS vs the generator's own calibration targets
# ============================================================
# The generator calibrates background std directly to EEG_BG_TARGET_RANGE / IEEG_BG_TARGET_RANGE
# (both already in uV), sampled per subject -- so the correct reference here is that exact
# range, not a generic literature band.
print("\n" + "=" * 100)
print("CELL 4 -- CONTINUOUS BACKGROUND RMS")
print("=" * 100)

def rms(X):
    return np.sqrt(np.mean(X**2, axis=2))

EEG_RMS = rms(EEG_UV)
FO_RMS = rms(FO_UV)

def summary(v):
    v = np.asarray(v, dtype=float).ravel()
    return np.mean(v), np.median(v), np.percentile(v, 5), np.percentile(v, 95)

eeg_mean, eeg_median, eeg_p5, eeg_p95 = summary(EEG_RMS)
fo_mean, fo_median, fo_p5, fo_p95 = summary(FO_RMS)

print(f"{'Metric':30s}{'Your value':20s}{'Reference':25s}{'Status'}")
print("-" * 90)
lo, hi = EEG_BG_TARGET_RANGE
for name, val in [("Mean RMS (uV)", eeg_mean), ("Median RMS (uV)", eeg_median)]:
    status = "PASS" if lo <= val <= hi else "WARN"
    print(f"{'EEG ' + name:30s}{val:<20.3f}{f'{lo:.0f}-{hi:.0f} uV RMS':25s}{status}")
print(f"{'EEG P5 RMS (uV)':30s}{eeg_p5:<20.3f}{'Distribution only':25s}INFO")
print(f"{'EEG P95 RMS (uV)':30s}{eeg_p95:<20.3f}{'Distribution only':25s}INFO")

lo, hi = IEEG_BG_TARGET_RANGE
for name, val in [("Mean RMS (uV)", fo_mean), ("Median RMS (uV)", fo_median)]:
    status = "PASS" if lo <= val <= hi else "WARN"
    print(f"{'FO ' + name:30s}{val:<20.3f}{f'{lo:.0f}-{hi:.0f} uV RMS':25s}{status}")
print(f"{'FO P5 RMS (uV)':30s}{fo_p5:<20.3f}{'Distribution only':25s}INFO")
print(f"{'FO P95 RMS (uV)':30s}{fo_p95:<20.3f}{'Distribution only':25s}INFO")



CELL 4 -- CONTINUOUS BACKGROUND RMS
Metric                        Your value          Reference                Status
------------------------------------------------------------------------------------------
EEG Mean RMS (uV)             8.200               12-20 uV RMS             WARN
EEG Median RMS (uV)           4.119               12-20 uV RMS             WARN
EEG P5 RMS (uV)               1.351               Distribution only        INFO
EEG P95 RMS (uV)              43.969              Distribution only        INFO
FO Mean RMS (uV)              89.493              70-130 uV RMS            PASS
FO Median RMS (uV)            84.622              70-130 uV RMS            PASS
FO P5 RMS (uV)                31.382              Distribution only        INFO
FO P95 RMS (uV)               159.213             Distribution only        INFO


In [5]:
# ============================================================
# CELL 5 -- EEG relative band power (+ relative alpha target)
# ============================================================
print("\n" + "=" * 100)
print("CELL 5 -- EEG RELATIVE BAND POWER")
print("=" * 100)

def band_power(x, lo, hi):
    f, p = signal.welch(x, fs=FS, nperseg=min(4096, len(x)))
    mask = (f >= lo) & (f < hi)
    return _trapz(p[mask], f[mask])

rows = []
for s in range(EEG.shape[0]):
    for c in range(EEG.shape[1]):
        x = EEG[s, c]
        total = band_power(x, 1, 80)
        values = {band: band_power(x, lo, hi) / (total + 1e-30) for band, (lo, hi) in BANDS.items()}
        rows.append({"subject": s, "channel": c, **values})
band_df = pd.DataFrame(rows)

print(f"{'Band':15s}{'Your mean':20s}{'Your median':20s}{'Reference range':25s}Status")
print("-" * 100)
for band in BANDS:
    val = band_df[band].mean()
    if band == "alpha":
        lo, hi = TARGETS["eeg_relative_alpha"]
        status = "PASS" if lo <= val <= hi else "WARN"
        ref = f"{lo:.2f}-{hi:.2f} (generator target)"
    else:
        status = "INFO"
        ref = f"{BANDS[band][0]}-{BANDS[band][1]} Hz band"
    print(f"{band:15s}{val:<20.4f}{band_df[band].median():<20.4f}{ref:25s}{status}")



CELL 5 -- EEG RELATIVE BAND POWER
Band           Your mean           Your median         Reference range          Status
----------------------------------------------------------------------------------------------------
delta          0.3801              0.3756              1-4 Hz band              INFO
theta          0.1824              0.1649              4-8 Hz band              INFO
alpha          0.2017              0.1343              0.15-0.30 (generator target)PASS
beta           0.1048              0.0955              13-30 Hz band            INFO
gamma          0.1242              0.0946              30-80 Hz band            INFO


In [6]:
# ============================================================
# CELL 6 -- EEG alpha peak frequency
# ============================================================
print("\n" + "=" * 100)
print("CELL 6 -- EEG ALPHA PEAK")
print("=" * 100)

alpha_peaks = []
for s in range(EEG.shape[0]):
    for c in range(EEG.shape[1]):
        f, p = signal.welch(EEG[s, c], fs=FS, nperseg=min(4096, EEG.shape[2]))
        mask = (f >= 8) & (f <= 13)
        if np.any(mask):
            alpha_peaks.append(f[mask][np.argmax(p[mask])])
alpha_peaks = np.asarray(alpha_peaks)

mean_alpha = alpha_peaks.mean()
median_alpha = np.median(alpha_peaks)
fraction_valid = np.mean((alpha_peaks >= 8) & (alpha_peaks <= 13))

print(f"{'Parameter':35s}{'Your value':20s}{'Reference':25s}Status")
print("-" * 100)
print(f"{'Mean alpha peak (Hz)':35s}{mean_alpha:<20.3f}{'8-13 Hz':25s}{'PASS' if 8 <= mean_alpha <= 13 else 'WARN'}")
print(f"{'Median alpha peak (Hz)':35s}{median_alpha:<20.3f}{'8-13 Hz':25s}{'PASS' if 8 <= median_alpha <= 13 else 'WARN'}")
print(f"{'Fraction within 8-13 Hz':35s}{fraction_valid:<20.3f}{'>80% preferred':25s}{'PASS' if fraction_valid >= .8 else 'WARN'}")
print("\nNote: the generator itself narrows its alpha oscillator to 8.5-11.5 Hz (CELL 1, "
      "v13 note) specifically to keep the measured peak inside the broader 8-13 Hz clinical band.")



CELL 6 -- EEG ALPHA PEAK
Parameter                          Your value          Reference                Status
----------------------------------------------------------------------------------------------------
Mean alpha peak (Hz)               9.547               8-13 Hz                  PASS
Median alpha peak (Hz)             9.521               8-13 Hz                  PASS
Fraction within 8-13 Hz            1.000               >80% preferred           PASS

Note: the generator itself narrows its alpha oscillator to 8.5-11.5 Hz (CELL 1, v13 note) specifically to keep the measured peak inside the broader 8-13 Hz clinical band.


In [7]:
# ============================================================
# CELL 7 -- 1/f aperiodic spectral slope
# ============================================================
print("\n" + "=" * 100)
print("CELL 7 -- 1/f SPECTRAL SLOPE")
print("=" * 100)

def get_slope(x):
    f, p = signal.welch(x, fs=FS, nperseg=min(4096, len(x)))
    mask = (f >= 2) & (f <= 40) & (p > 0)
    return np.polyfit(np.log10(f[mask]), np.log10(p[mask]), 1)[0]

eeg_slopes = np.array([get_slope(EEG[s, c]) for s in range(EEG.shape[0]) for c in range(EEG.shape[1])])
fo_slopes = np.array([get_slope(FO[s, c]) for s in range(FO.shape[0]) for c in range(FO.shape[1])])

def summarize_slopes(values, name):
    neg_frac = np.mean(values < 0)
    status = "PASS" if neg_frac >= 0.90 else ("WARN" if neg_frac >= 0.75 else "FAIL")
    print(f"\n{name}")
    print(f"{'Mean slope':35s}{np.mean(values):<20.4f}{'Negative (aperiodic 1/f)':35s}{'PASS' if np.mean(values) < 0 else 'WARN'}")
    print(f"{'Median slope':35s}{np.median(values):<20.4f}{'Negative (aperiodic 1/f)':35s}{'PASS' if np.median(values) < 0 else 'WARN'}")
    print(f"{'Fraction negative':35s}{neg_frac:<20.3f}{'>=0.90 preferred':35s}{status}")

summarize_slopes(eeg_slopes, "SCALP EEG")
summarize_slopes(fo_slopes, "FO-iEEG")



CELL 7 -- 1/f SPECTRAL SLOPE

SCALP EEG
Mean slope                         -1.5599             Negative (aperiodic 1/f)           PASS
Median slope                       -1.5652             Negative (aperiodic 1/f)           PASS
Fraction negative                  1.000               >=0.90 preferred                   PASS

FO-iEEG
Mean slope                         -1.8712             Negative (aperiodic 1/f)           PASS
Median slope                       -1.8949             Negative (aperiodic 1/f)           PASS
Fraction negative                  1.000               >=0.90 preferred                   PASS


In [8]:
# ============================================================
# CELL 8 -- Shared IED-window extraction helper
# ============================================================
# Used by every downstream IED metric cell. Extracts, for every valid onset in IED_ONSETS,
# a baseline-relative event window on both FO and EEG, the strongest FO contact for that
# event, and the matching EEG channel (same electrode index range is not meaningful across
# modalities, so EEG uses its own per-event peak channel).

PRE_SEC = 0.050
EVENT_SEC = 0.200
PRE = int(round(PRE_SEC * FS))
EVENT_N = int(round(EVENT_SEC * FS))

def extract_ied_windows(EEG_UV, FO_UV, ied_onsets, pre=PRE, event_n=EVENT_N):
    records = []
    n_subjects_ = EEG_UV.shape[0]
    for s in range(n_subjects_):
        for onset in ied_onsets[s]:
            onset = int(onset)
            a, b = onset - pre, onset + event_n
            if a < 0 or b > FO_UV.shape[2] or b > EEG_UV.shape[2]:
                continue
            fw = FO_UV[s, :, a:b]
            ew = EEG_UV[s, :, a:b]
            f_base = np.median(fw[:, :pre], axis=1)
            e_base = np.median(ew[:, :pre], axis=1)
            fx = fw - f_base[:, None]
            ex = ew - e_base[:, None]
            f_bg_rms = np.sqrt(np.mean((fw[:, :pre] - f_base[:, None]) ** 2, axis=1))
            f_peak_per_ch = np.max(np.abs(fx[:, pre:]), axis=1)
            e_peak_per_ch = np.max(np.abs(ex[:, pre:]), axis=1)
            fch = int(np.argmax(f_peak_per_ch))
            ech = int(np.argmax(e_peak_per_ch))
            records.append(dict(
                subject=s, onset=onset,
                fo_peak=float(f_peak_per_ch[fch]), fo_ch=fch,
                fo_bg_rms=float(f_bg_rms[fch]),
                eeg_peak=float(e_peak_per_ch[ech]), eeg_ch=ech,
                fo_trace=fx[fch], eeg_trace=ex[ech],
                fo_all_peak=f_peak_per_ch, eeg_all_peak=e_peak_per_ch,
            ))
    return records

IED_EVENTS = extract_ied_windows(EEG_UV, FO_UV, IED_ONSETS)
print(f"Extracted {len(IED_EVENTS)} valid IED event windows "
      f"(out of {sum(len(x) for x in IED_ONSETS)} total onsets).")


Extracted 1800 valid IED event windows (out of 1800 total onsets).


In [9]:
# ============================================================
# CELL 9 -- IED peak amplitude + amplitude CV
# ============================================================
print("\n" + "=" * 100)
print("CELL 9 -- IED PEAK AMPLITUDE & VARIABILITY")
print("=" * 100)

IED_AMPS = np.array([e["fo_peak"] for e in IED_EVENTS], dtype=float)
IED_EEG_AMPS = np.array([e["eeg_peak"] for e in IED_EVENTS], dtype=float)

if len(IED_AMPS) < 2:
    raise RuntimeError("Too few valid IED amplitude measurements -- check CSV_PATH / ied_flag.")

lo, hi = TARGETS["fo_peak_amp_uv"]
print(f"{'Metric':40s}{'Your value':20s}{'Reference':25s}Status")
print("-" * 105)
print(f"{'Valid IED events':40s}{len(IED_AMPS):<20d}{'Dataset dependent':25s}INFO")
med_fo_amp = np.median(IED_AMPS)
print(f"{'FO IED peak median (uV)':40s}{med_fo_amp:<20.2f}{f'{lo:.0f}-{hi:.0f} uV':25s}{'PASS' if lo <= med_fo_amp <= hi else 'WARN'}")
print(f"{'FO IED peak mean (uV)':40s}{np.mean(IED_AMPS):<20.2f}{'Distribution':25s}INFO")
print(f"{'FO IED peak P5 (uV)':40s}{np.percentile(IED_AMPS,5):<20.2f}{'Distribution':25s}INFO")
print(f"{'FO IED peak P95 (uV)':40s}{np.percentile(IED_AMPS,95):<20.2f}{'Distribution':25s}INFO")
print(f"{'EEG IED peak median (uV)':40s}{np.median(IED_EEG_AMPS):<20.2f}{'No fixed target':25s}INFO")

amp_cv = np.std(IED_AMPS, ddof=1) / max(np.mean(IED_AMPS), 1e-12)
lo, hi = TARGETS["ied_amp_cv"]
print(f"{'FO IED amplitude CV':40s}{amp_cv:<20.3f}{f'{lo:.2f}-{hi:.2f}':25s}{'PASS' if lo <= amp_cv <= hi else 'WARN'}")



CELL 9 -- IED PEAK AMPLITUDE & VARIABILITY
Metric                                  Your value          Reference                Status
---------------------------------------------------------------------------------------------------------
Valid IED events                        1800                Dataset dependent        INFO
FO IED peak median (uV)                 992.23              200-1000 uV              PASS
FO IED peak mean (uV)                   1173.43             Distribution             INFO
FO IED peak P5 (uV)                     301.31              Distribution             INFO
FO IED peak P95 (uV)                    2634.45             Distribution             INFO
EEG IED peak median (uV)                74.84               No fixed target          INFO
FO IED amplitude CV                     0.683               0.25-0.80                PASS


In [10]:
# ============================================================
# CELL 10 -- IED FWHM (separate EEG and FO targets)
# ============================================================
# v18 gives EEG its own, independently sharper waveform (EEG_SHARPEN, CELL 1 of the
# generator), so EEG and FO have DIFFERENT FWHM targets and must be measured/reported
# separately rather than pooled together as one 'IED width' number.
print("\n" + "=" * 100)
print("CELL 10 -- IED FWHM (EEG vs FO)")
print("=" * 100)

def fwhm_ms(trace, fs):
    peak_idx = int(np.argmax(np.abs(trace)))
    peak = trace[peak_idx]
    amp = abs(peak)
    if not np.isfinite(amp) or amp <= 0:
        return np.nan
    half = 0.5 * amp
    left = peak_idx
    while left > 0 and abs(trace[left]) >= half:
        left -= 1
    right = peak_idx
    while right < len(trace) - 1 and abs(trace[right]) >= half:
        right += 1
    return (right - left) / fs * 1000.0

fo_fwhm = np.array([fwhm_ms(e["fo_trace"], FS) for e in IED_EVENTS])
eeg_fwhm = np.array([fwhm_ms(e["eeg_trace"], FS) for e in IED_EVENTS])
fo_fwhm = fo_fwhm[np.isfinite(fo_fwhm)]
eeg_fwhm = eeg_fwhm[np.isfinite(eeg_fwhm)]

print(f"{'Metric':35s}{'Median':14s}{'P5':12s}{'P95':12s}{'SD':12s}{'Reference':20s}Status")
print("-" * 120)
for name, v, key in [("EEG FWHM (ms)", eeg_fwhm, "eeg_fwhm_ms"), ("FO FWHM (ms)", fo_fwhm, "fo_fwhm_ms")]:
    lo, hi = TARGETS[key]
    fwhm_med = np.median(v)
    status = "PASS" if lo <= fwhm_med <= hi else "WARN"
    print(f"{name:35s}{fwhm_med:<14.2f}{np.percentile(v,5):<12.2f}{np.percentile(v,95):<12.2f}"
          f"{np.std(v):<12.2f}{f'{lo:.0f}-{hi:.0f} ms':20s}{status}")



CELL 10 -- IED FWHM (EEG vs FO)
Metric                             Median        P5          P95         SD          Reference           Status
------------------------------------------------------------------------------------------------------------------------
EEG FWHM (ms)                      15.00         10.00       30.00       12.48       15-40 ms            PASS
FO FWHM (ms)                       20.00         10.00       35.00       13.15       10-30 ms            PASS


In [11]:
# ============================================================
# CELL 11 -- IED rate per subject (+ event-count sanity check)
# ============================================================
print("\n" + "=" * 100)
print("CELL 11 -- IED RATE PER SUBJECT")
print("=" * 100)

subject_counts = np.array([len(x) for x in IED_ONSETS], dtype=int)
subject_rate = subject_counts / (n_samples_found / FS / 60.0)

lo, hi = TARGETS["ied_rate_per_min"]
print(f"{'Subject':10s}{'IED count':15s}{'Expected':15s}{'Rate (/min)':15s}Status")
print("-" * 70)
for s in range(n_subjects_found):
    status_n = "PASS" if subject_counts[s] == N_IED_EXPECTED_PER_SUBJECT else "WARN"
    status_r = "PASS" if lo <= subject_rate[s] <= hi else "WARN"
    print(f"{s+1:<10d}{subject_counts[s]:<15d}{N_IED_EXPECTED_PER_SUBJECT:<15d}{subject_rate[s]:<15.2f}{status_n}/{status_r}")

print(f"\n{'Metric':40s}{'Your value':20s}{'Reference':25s}Status")
print("-" * 100)
print(f"{'Mean IED rate (/min)':40s}{np.mean(subject_rate):<20.3f}{f'{lo:.0f}-{hi:.0f} /min':25s}{'PASS' if lo <= np.mean(subject_rate) <= hi else 'WARN'}")
print(f"{'Median IED rate (/min)':40s}{np.median(subject_rate):<20.3f}{f'{lo:.0f}-{hi:.0f} /min':25s}INFO")
print(f"{'Subjects with expected count ({0})'.format(N_IED_EXPECTED_PER_SUBJECT):40s}"
      f"{int(np.sum(subject_counts == N_IED_EXPECTED_PER_SUBJECT)):<20d}{N_SUBJECTS_EXPECTED:<25}"
      f"{'PASS' if np.all(subject_counts == N_IED_EXPECTED_PER_SUBJECT) else 'WARN'}")



CELL 11 -- IED RATE PER SUBJECT
Subject   IED count      Expected       Rate (/min)    Status
----------------------------------------------------------------------
1         100            100            6.67           PASS/PASS
2         100            100            6.67           PASS/PASS
3         100            100            6.67           PASS/PASS
4         100            100            6.67           PASS/PASS
5         100            100            6.67           PASS/PASS
6         100            100            6.67           PASS/PASS
7         100            100            6.67           PASS/PASS
8         100            100            6.67           PASS/PASS
9         100            100            6.67           PASS/PASS
10        100            100            6.67           PASS/PASS
11        100            100            6.67           PASS/PASS
12        100            100            6.67           PASS/PASS
13        100            100            6.67          

In [12]:
# ============================================================
# CELL 12 -- EEG <-> FO propagation lag (cross-correlation, per event)
# ============================================================
print("\n" + "=" * 100)
print("CELL 12 -- EEG-FO PROPAGATION DELAY")
print("=" * 100)

def positive_lag_ms(a, b, fs, max_lag_ms=100.0):
    a = a - np.mean(a); b = b - np.mean(b)
    sa, sb = np.std(a), np.std(b)
    if not np.isfinite(sa) or not np.isfinite(sb) or sa <= 1e-12 or sb <= 1e-12:
        return np.nan
    c = correlate(b, a, mode="full")
    lags = np.arange(-len(a) + 1, len(a))
    max_lag_samp = int(round(max_lag_ms / 1000.0 * fs))
    mask = (lags >= 0) & (lags <= max_lag_samp)
    if not np.any(mask):
        return np.nan
    best = lags[mask][np.argmax(c[mask])]
    return best / fs * 1000.0

lags = []
for e in IED_EVENTS:
    lag = positive_lag_ms(e["eeg_trace"], e["fo_trace"], FS)
    if np.isfinite(lag):
        lags.append(lag)
lags = np.asarray(lags)

lo, hi = TARGETS["propagation_lag_ms"]
lag_med = np.median(lags) if len(lags) else np.nan
print(f"{'Metric':40s}{'Your value':20s}{'Reference':25s}Status")
print("-" * 100)
print(f"{'Median EEG->FO lag (ms)':40s}{lag_med:<20.2f}{f'{lo:.0f}-{hi:.0f} ms':25s}"
      f"{'PASS' if np.isfinite(lag_med) and lo <= lag_med <= hi else 'WARN'}")
print(f"{'Mean lag (ms)':40s}{np.mean(lags):<20.2f}{'Distribution':25s}INFO")
print(f"{'Lag P5-P95 (ms)':40s}{f'{np.percentile(lags,5):.1f}-{np.percentile(lags,95):.1f}':<20s}{'Distribution':25s}INFO")
print(f"{'Events measured':40s}{len(lags):<20d}{'of ' + str(len(IED_EVENTS)):25s}INFO")



CELL 12 -- EEG-FO PROPAGATION DELAY
Metric                                  Your value          Reference                Status
----------------------------------------------------------------------------------------------------
Median EEG->FO lag (ms)                 55.00               5-30 ms                  WARN
Mean lag (ms)                           49.93               Distribution             INFO
Lag P5-P95 (ms)                         0.0-100.0           Distribution             INFO
Events measured                         1800                of 1800                  INFO


In [13]:
# ============================================================
# CELL 13 -- FO/EEG peak ratio + FO hemispheric lateralisation
# ============================================================
print("\n" + "=" * 100)
print("CELL 13 -- FO/EEG PEAK RATIO & FO LATERALISATION")
print("=" * 100)

ratios = IED_AMPS / np.maximum(IED_EEG_AMPS, 1e-9)
lo, hi = TARGETS["fo_eeg_peak_ratio"]
med_ratio = np.median(ratios)
print(f"{'Metric':40s}{'Your value':20s}{'Reference':25s}Status")
print("-" * 100)
print(f"{'FO/EEG peak ratio (median)':40s}{med_ratio:<20.2f}{f'{lo:.0f}-{hi:.0f}':25s}{'PASS' if lo <= med_ratio <= hi else 'WARN'}")

# Lateralisation: for each event, compare the max FO peak on the SAME hemisphere as the
# strongest contact vs the max FO peak on the OPPOSITE hemisphere.
fo_idx_to_ch = {i: ch for i, ch in enumerate(IEEG_CHANNELS)}
lat_ratios = []
for e in IED_EVENTS:
    fch = e["fo_ch"]
    same_hemi = IEEG_HEMI[fo_idx_to_ch[fch]]
    all_peaks = e["fo_all_peak"]
    same_mask = np.array([IEEG_HEMI[fo_idx_to_ch[i]] == same_hemi for i in range(len(all_peaks))])
    same_max = np.max(all_peaks[same_mask])
    cross_max = np.max(all_peaks[~same_mask]) if np.any(~same_mask) else np.nan
    if np.isfinite(cross_max) and cross_max > 1e-9:
        lat_ratios.append(same_max / cross_max)
lat_ratios = np.asarray(lat_ratios)

lo, hi = TARGETS["fo_lateralisation_ratio"]
med_lat = np.median(lat_ratios) if len(lat_ratios) else np.nan
hi_str = "inf" if not np.isfinite(hi) else f"{hi:.0f}"
print(f"{'FO L/R lateralisation (median)':40s}{med_lat:<20.2f}{f'>= {lo:.0f}':25s}"
      f"{'PASS' if np.isfinite(med_lat) and med_lat >= lo else 'WARN'}")
print(f"{'Events with valid lateralisation':40s}{len(lat_ratios):<20d}{'of ' + str(len(IED_EVENTS)):25s}INFO")



CELL 13 -- FO/EEG PEAK RATIO & FO LATERALISATION
Metric                                  Your value          Reference                Status
----------------------------------------------------------------------------------------------------
FO/EEG peak ratio (median)              12.96               10-40                    PASS
FO L/R lateralisation (median)          5.80                >= 5                     PASS
Events with valid lateralisation        1800                of 1800                  INFO


In [14]:
# ============================================================
# CELL 14 -- Scalp IED detectability (AUC via RandomForest, matching the generator's
# own described "STEP 8" feature set)
# ============================================================
# The generator's changelog (v15) describes the real validator's scalp classifier as using:
# RMS, std, abs-max, line-length, kurtosis, and 5 band-power means, fed to a
# RandomForestClassifier, distinguishing IED-window features from randomly sampled non-IED
# background windows. Reproduced here directly instead of a coarse peak/background-std proxy.
print("\n" + "=" * 100)
print("CELL 14 -- SCALP EEG IED DETECTABILITY (AUC)")
print("=" * 100)

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score
from sklearn.metrics import roc_auc_score

WIN = int(round(0.32 * FS))  # 320 ms analysis window, matching the generator's own framing

def window_features(x):
    rms_ = np.sqrt(np.mean(x**2))
    std_ = np.std(x)
    absmax_ = np.max(np.abs(x))
    line_len = np.sum(np.abs(np.diff(x)))
    kurt = stats.kurtosis(x)
    feats = [rms_, std_, absmax_, line_len, kurt]
    for lo, hi in BANDS.values():
        feats.append(band_power(x, lo, hi))
    return feats

rng = np.random.default_rng(42)
X_feat, y_lab = [], []
for s in range(n_subjects_found):
    onsets = IED_ONSETS[s]
    exclude = np.zeros(n_samples_found, dtype=bool)
    for t in onsets:
        a, b = max(0, t - WIN), min(n_samples_found, t + WIN)
        exclude[a:b] = True
    # positive windows: onset -> onset+WIN, best scalp channel per event
    for t in onsets:
        a, b = int(t), int(t) + WIN
        if b > n_samples_found:
            continue
        seg = EEG_UV[s, :, a:b]
        ch = int(np.argmax(np.max(np.abs(seg), axis=1)))
        X_feat.append(window_features(seg[ch]))
        y_lab.append(1)
    # negative windows: same count, randomly placed, non-overlapping with any IED
    n_pos = len(onsets)
    n_found = 0
    tries = 0
    while n_found < n_pos and tries < n_pos * 100:
        tries += 1
        t = rng.integers(0, n_samples_found - WIN)
        if exclude[t:t + WIN].any():
            continue
        seg = EEG_UV[s, :, t:t + WIN]
        ch = int(rng.integers(0, seg.shape[0]))
        X_feat.append(window_features(seg[ch]))
        y_lab.append(0)
        n_found += 1

X_feat = np.asarray(X_feat)
y_lab = np.asarray(y_lab)

clf = RandomForestClassifier(n_estimators=200, max_depth=6, random_state=0)
cv_auc = cross_val_score(clf, X_feat, y_lab, cv=5, scoring="roc_auc")
auc = float(np.mean(cv_auc))

lo, hi = TARGETS["scalp_auc"]
print(f"{'Metric':40s}{'Your value':20s}{'Reference':25s}Status")
print("-" * 100)
print(f"{'Scalp IED AUC (5-fold CV mean)':40s}{auc:<20.3f}{f'{lo:.2f}-{hi:.2f}':25s}{'PASS' if lo <= auc <= hi else 'WARN'}")
print(f"{'AUC fold SD':40s}{np.std(cv_auc):<20.3f}{'Distribution':25s}INFO")
print(f"{'Windows used (pos/neg)':40s}{f'{int(y_lab.sum())}/{int((1-y_lab).sum())}':<20s}{'Balanced':25s}INFO")
print("\nAUC near 1.0 means the injected spike (or an artefact correlated with it, e.g. an "
      "always-present background suppression dip) is trivially separable from background -- "
      "not physiologically realistic. AUC near 0.5 means it's undetectable even to a "
      "purpose-built classifier. The 0.50-0.75 target represents 'detectable but not trivial', "
      "matching real interictal scalp spike detection difficulty.")



CELL 14 -- SCALP EEG IED DETECTABILITY (AUC)
Metric                                  Your value          Reference                Status
----------------------------------------------------------------------------------------------------
Scalp IED AUC (5-fold CV mean)          0.996               0.50-0.75                WARN
AUC fold SD                             0.003               Distribution             INFO
Windows used (pos/neg)                  1800/1800           Balanced                 INFO

AUC near 1.0 means the injected spike (or an artefact correlated with it, e.g. an always-present background suppression dip) is trivially separable from background -- not physiologically realistic. AUC near 0.5 means it's undetectable even to a purpose-built classifier. The 0.50-0.75 target represents 'detectable but not trivial', matching real interictal scalp spike detection difficulty.


In [15]:
# ============================================================
# CELL 15 -- FO line-length ratio & ERSP gain (event window vs pre-event baseline)
# ============================================================
print("\n" + "=" * 100)
print("CELL 15 -- FO LINE-LENGTH RATIO & ERSP GAIN")
print("=" * 100)

def line_length(x):
    return np.sum(np.abs(np.diff(x)))

ll_ratios, ersp_gains = [], []
for e in IED_EVENTS:
    trace = e["fo_trace"]                       # already baseline-subtracted, length pre+event_n
    pre_seg = trace[:PRE]
    post_seg = trace[PRE:]
    ll_pre = line_length(pre_seg)
    ll_post = line_length(post_seg)
    if ll_pre > 1e-9:
        ll_ratios.append(ll_post / ll_pre)
    pre_power = np.mean(pre_seg ** 2)
    post_power = np.mean(post_seg ** 2)
    if pre_power > 1e-9:
        ersp_gains.append(post_power / pre_power)

ll_ratios = np.asarray(ll_ratios)
ersp_gains = np.asarray(ersp_gains)

lo, hi = TARGETS["fo_line_length_ratio"]
med_ll = np.median(ll_ratios)
print(f"{'Metric':40s}{'Your value':20s}{'Reference':25s}Status")
print("-" * 100)
print(f"{'FO line-length ratio (median)':40s}{med_ll:<20.3f}{f'{lo:.1f}-{hi:.1f}':25s}{'PASS' if lo <= med_ll <= hi else 'WARN'}")

lo, hi = TARGETS["fo_ersp_gain"]
med_ersp = np.median(ersp_gains)
print(f"{'FO ERSP gain (median, power ratio)':40s}{med_ersp:<20.3f}{f'>= {lo:.0f}':25s}{'PASS' if med_ersp >= lo else 'WARN'}")



CELL 15 -- FO LINE-LENGTH RATIO & ERSP GAIN
Metric                                  Your value          Reference                Status
----------------------------------------------------------------------------------------------------
FO line-length ratio (median)           8.422               1.8-8.0                  WARN
FO ERSP gain (median, power ratio)      28.470              >= 3                     PASS


In [16]:
# ============================================================
# CELL 16 -- 5-minute block variability (non-stationarity, descriptive)
# ============================================================
print("\n" + "=" * 100)
print("CELL 16 -- BLOCK-LEVEL VARIABILITY")
print("=" * 100)

BLOCK_SEC = min(300.0, n_samples_found / FS / 2.0)   # fall back to 2 blocks if the recording
BLOCK = int(round(BLOCK_SEC * FS))                     # is shorter than 300s (e.g. a quick test run)
N_BLOCKS = n_samples_found // BLOCK if BLOCK > 0 else 0

if N_BLOCKS < 2:
    print("Recording too short relative to BLOCK_SEC to compute block-level variability "
          f"(got {N_BLOCKS} block(s)). Skipping.")
    eeg_cv_list, fo_cv_list = [np.nan], [np.nan]
else:
    rows = []
    for s in range(n_subjects_found):
        for k in range(N_BLOCKS):
            a, b = k * BLOCK, (k + 1) * BLOCK
            eeg_rms = float(np.sqrt(np.mean(EEG_UV[s, :, a:b] ** 2)))
            fo_rms = float(np.sqrt(np.mean(FO_UV[s, :, a:b] ** 2)))
            rows.append({"subject": s, "block": k, "EEG_RMS": eeg_rms, "FO_RMS": fo_rms})
    block_df = pd.DataFrame(rows)

    eeg_cv_list, fo_cv_list = [], []
    for s, g in block_df.groupby("subject"):
        eeg_cv_list.append(np.std(g.EEG_RMS, ddof=1) / max(np.mean(g.EEG_RMS), 1e-12))
        fo_cv_list.append(np.std(g.FO_RMS, ddof=1) / max(np.mean(g.FO_RMS), 1e-12))

    print(f"{'Metric':40s}{'Median subject CV':20s}{'Interpretation':30s}Status")
    print("-" * 105)
    print(f"{'Block EEG RMS variability':40s}{np.median(eeg_cv_list):<20.3f}{'Moderate nonstationarity':30s}INFO")
    print(f"{'Block FO RMS variability':40s}{np.median(fo_cv_list):<20.3f}{'Moderate nonstationarity':30s}INFO")
    print(f"{'Block length (s) / blocks used':40s}{f'{BLOCK_SEC:.0f}s / {N_BLOCKS}':<20s}{'900s -> 300s blocks = 3':30s}INFO")
    print("\nNo universal clinical CV cutoff is imposed. Near-zero CV is flagged as suspiciously "
          "stationary; very large CV should be inspected manually.")



CELL 16 -- BLOCK-LEVEL VARIABILITY
Metric                                  Median subject CV   Interpretation                Status
---------------------------------------------------------------------------------------------------------
Block EEG RMS variability               0.061               Moderate nonstationarity      INFO
Block FO RMS variability                0.030               Moderate nonstationarity      INFO
Block length (s) / blocks used          300s / 3            900s -> 300s blocks = 3       INFO

No universal clinical CV cutoff is imposed. Near-zero CV is flagged as suspiciously stationary; very large CV should be inspected manually.


In [17]:
# ============================================================
# CELL 17 -- Inter-subject variability (descriptive)
# ============================================================
print("\n" + "=" * 100)
print("CELL 17 -- INTER-SUBJECT VARIABILITY")
print("=" * 100)

def subject_rms(X):
    return np.array([np.sqrt(np.mean(X[s] ** 2)) for s in range(X.shape[0])])

eeg_subject_rms = subject_rms(EEG_UV)
fo_subject_rms = subject_rms(FO_UV)

def cv(x):
    return np.std(x, ddof=1) / max(np.mean(x), 1e-12)

lo, hi = IEEG_BG_TARGET_RANGE
print(f"{'Parameter':35s}{'Your value':20s}{'Reference':30s}Status")
print("-" * 100)
print(f"{'EEG RMS inter-subject CV':35s}{cv(eeg_subject_rms):<20.3f}{'Dataset-dependent':30s}INFO")
print(f"{'FO RMS inter-subject CV':35s}{cv(fo_subject_rms):<20.3f}{'Dataset-dependent':30s}INFO")
print(f"{'EEG RMS median (uV)':35s}{np.median(eeg_subject_rms):<20.3f}{'Distribution':30s}INFO")
print(f"{'FO RMS median (uV)':35s}{np.median(fo_subject_rms):<20.3f}{f'{lo:.0f}-{hi:.0f} uV target':30s}"
      f"{'PASS' if lo <= np.median(fo_subject_rms) <= hi else 'WARN'}")



CELL 17 -- INTER-SUBJECT VARIABILITY
Parameter                          Your value          Reference                     Status
----------------------------------------------------------------------------------------------------
EEG RMS inter-subject CV           0.147               Dataset-dependent             INFO
FO RMS inter-subject CV            0.127               Dataset-dependent             INFO
EEG RMS median (uV)                15.104              Distribution                  INFO
FO RMS median (uV)                 99.968              70-130 uV target              PASS


In [18]:
# ============================================================
# CELL 18 -- FINAL SUMMARY: generator-target realism report
# ============================================================
print("\n" + "=" * 125)
print("FINAL REALISM REPORT -- validated against 01_dataset_generation_v18.ipynb's own stated targets")
print("=" * 125)
print(f"Sampling frequency: {FS:.0f} Hz | Duration/subject: {n_samples_found/FS/60:.1f} min | "
      f"Subjects: {n_subjects_found} | Units: microvolts")
print("-" * 125)
print(f"{'#':<4}{'CHECK':<38}{'YOUR VALUE':<20}{'REFERENCE':<26}{'STATUS':<10}")
print("-" * 125)

results = []
def add(n, check, value, ref, status):
    results.append(status)
    print(f"{n:<4}{check:<38}{str(value):<20}{ref:<26}{status:<10}")

# structure / integrity
add(1, "Subjects", n_subjects_found, str(N_SUBJECTS_EXPECTED), "PASS" if n_subjects_found == N_SUBJECTS_EXPECTED else "FAIL")
add(2, "EEG channels", EEG.shape[1], str(N_EEG), "PASS" if EEG.shape[1] == N_EEG else "FAIL")
add(3, "FO channels", FO.shape[1], str(N_FO), "PASS" if FO.shape[1] == N_FO else "FAIL")
add(4, "Sampling frequency", f"{FS:.0f} Hz", "200 Hz", "PASS" if FS == 200 else "FAIL")
add(5, "Duration/subject", f"{n_samples_found/FS/60:.1f} min", "15 min",
    "PASS" if abs(n_samples_found/FS/60 - 15) < 0.05 else "FAIL")
add(6, "NaN/Inf/dead channels", f"{EEG_integrity['nan']+FO_integrity['nan']} / "
    f"{EEG_integrity['inf']+FO_integrity['inf']} / {EEG_integrity['dead']+FO_integrity['dead']}",
    "0 / 0 / 0", "PASS" if (EEG_integrity['nan']+FO_integrity['nan']+EEG_integrity['inf']
    +FO_integrity['inf']+EEG_integrity['dead']+FO_integrity['dead']) == 0 else "FAIL")

# physiology / generator targets
lo, hi = EEG_BG_TARGET_RANGE
add(7, "EEG background RMS (median)", f"{eeg_median:.1f} uV", f"{lo:.0f}-{hi:.0f} uV",
    "PASS" if lo <= eeg_median <= hi else "WARN")
lo, hi = IEEG_BG_TARGET_RANGE
add(8, "FO background RMS (median)", f"{fo_median:.1f} uV", f"{lo:.0f}-{hi:.0f} uV",
    "PASS" if lo <= fo_median <= hi else "WARN")
add(9, "EEG alpha peak (median)", f"{median_alpha:.2f} Hz", "8-13 Hz",
    "PASS" if 8 <= median_alpha <= 13 else "WARN")
lo, hi = TARGETS["eeg_relative_alpha"]
rel_alpha = band_df["alpha"].mean()
add(10, "EEG relative alpha power", f"{rel_alpha:.3f}", f"{lo:.2f}-{hi:.2f}",
    "PASS" if lo <= rel_alpha <= hi else "WARN")
add(11, "EEG 1/f slope (median)", f"{np.median(eeg_slopes):.3f}", "Negative",
    "PASS" if np.median(eeg_slopes) < 0 else "FAIL")
add(12, "FO 1/f slope (median)", f"{np.median(fo_slopes):.3f}", "Negative",
    "PASS" if np.median(fo_slopes) < 0 else "WARN")
lo, hi = TARGETS["fo_peak_amp_uv"]
add(13, "FO IED peak amplitude (median)", f"{np.median(IED_AMPS):.0f} uV", f"{lo:.0f}-{hi:.0f} uV",
    "PASS" if lo <= np.median(IED_AMPS) <= hi else "WARN")
lo, hi = TARGETS["ied_amp_cv"]
add(14, "FO IED amplitude CV", f"{amp_cv:.3f}", f"{lo:.2f}-{hi:.2f}",
    "PASS" if lo <= amp_cv <= hi else "WARN")
lo, hi = TARGETS["eeg_fwhm_ms"]
add(15, "EEG IED FWHM (median)", f"{np.median(eeg_fwhm):.1f} ms", f"{lo:.0f}-{hi:.0f} ms",
    "PASS" if lo <= np.median(eeg_fwhm) <= hi else "WARN")
lo, hi = TARGETS["fo_fwhm_ms"]
add(16, "FO IED FWHM (median)", f"{np.median(fo_fwhm):.1f} ms", f"{lo:.0f}-{hi:.0f} ms",
    "PASS" if lo <= np.median(fo_fwhm) <= hi else "WARN")
lo, hi = TARGETS["ied_rate_per_min"]
mean_rate = np.mean(subject_rate)
add(17, "IED rate (mean/subject)", f"{mean_rate:.2f} /min", f"{lo:.0f}-{hi:.0f} /min",
    "PASS" if lo <= mean_rate <= hi else "WARN")
lo, hi = TARGETS["propagation_lag_ms"]
add(18, "EEG-FO propagation lag (median)", f"{lag_med:.1f} ms" if np.isfinite(lag_med) else "N/A",
    f"{lo:.0f}-{hi:.0f} ms", "PASS" if np.isfinite(lag_med) and lo <= lag_med <= hi else "WARN")
lo, hi = TARGETS["fo_eeg_peak_ratio"]
ratio_med = np.median(ratios)
add(19, "FO/EEG peak ratio (median)", f"{ratio_med:.2f}", f"{lo:.0f}-{hi:.0f}",
    "PASS" if lo <= ratio_med <= hi else "WARN")
lo, _ = TARGETS["fo_lateralisation_ratio"]
add(20, "FO lateralisation L/R (median)", f"{med_lat:.2f}" if np.isfinite(med_lat) else "N/A",
    f">= {lo:.0f}", "PASS" if np.isfinite(med_lat) and med_lat >= lo else "WARN")
lo, hi = TARGETS["fo_line_length_ratio"]
add(21, "FO line-length ratio (median)", f"{med_ll:.2f}", f"{lo:.1f}-{hi:.1f}",
    "PASS" if lo <= med_ll <= hi else "WARN")
lo, _ = TARGETS["fo_ersp_gain"]
add(22, "FO ERSP gain (median)", f"{med_ersp:.2f}", f">= {lo:.0f}",
    "PASS" if med_ersp >= lo else "WARN")
lo, hi = TARGETS["scalp_auc"]
add(23, "Scalp IED detectability (AUC)", f"{auc:.3f}", f"{lo:.2f}-{hi:.2f}",
    "PASS" if lo <= auc <= hi else "WARN")

print("-" * 125)
pass_n = results.count("PASS")
warn_n = results.count("WARN")
fail_n = results.count("FAIL")
print(f"\nTOTAL: {pass_n} PASS / {warn_n} WARN / {fail_n} FAIL  (out of {len(results)} checks)")
print("\nInterpretation: PASS/WARN reflect the generator's OWN stated design targets (pulled "
      "directly from its changelog cells), not an independent clinical ground truth. FAIL is "
      "reserved for structural/integrity problems (wrong shape, NaN/Inf, wrong fs, non-negative "
      "1/f slope) that indicate something is actually broken, not just out of the preferred range.")



FINAL REALISM REPORT -- validated against 01_dataset_generation_v18.ipynb's own stated targets
Sampling frequency: 200 Hz | Duration/subject: 15.0 min | Subjects: 18 | Units: microvolts
-----------------------------------------------------------------------------------------------------------------------------
#   CHECK                                 YOUR VALUE          REFERENCE                 STATUS    
-----------------------------------------------------------------------------------------------------------------------------
1   Subjects                              18                  18                        PASS      
2   EEG channels                          20                  20                        PASS      
3   FO channels                           12                  12                        PASS      
4   Sampling frequency                    200 Hz              200 Hz                    PASS      
5   Duration/subject                      15.0 min            15 m